# **Part 1: Loading and Cleaning**

This notebook loads the raw SBA file, works out which loans can actually be used, cleans up what needs cleaning, and saves the result for the next notebooks.

Input: FOIA_7a_FY2020_Present_asof_260630.csv
Output: model_df_cleaned.csv

## **Stage 1:** Loading and Exploring data

In [1]:
# Importing the data 
import pandas as pd      # For loading and reshaping the loan data

df = pd.read_csv('FOIA_7a_FY2020_Present_asof_260630.csv')
df

C:\Users\Muhammad Mubashar\AppData\Local\Temp\ipykernel_23096\2069898143.py:4: DtypeWarning: Columns (0: ChargeOffDate) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('FOIA_7a_FY2020_Present_asof_260630.csv')


,AsOfDate,Program,LocationID,BorrName,BorrStreet,BorrCity,BorrState,BorrZip,BankName,BankFDICNumber,...,BusinessType,BusinessAge,LoanStatus,PaidInFullDate,ChargeOffDate,GrossChargeOffAmount,RevolverStatus,JobsSupported,CollateralInd,SoldSecMrktInd
0,2026-06-30,7A,29805,"AMERIPRO CONSTRUCTION SERVICES, INC.",1403 SENTRY LANE,Norristown,PA,19403,"TD Bank, National Association",18409.0,...,PARTNERSHIP,Unanswered,P I F,2024-11-30,NaN,0.0,Y,5.0,N,NaN
1,2026-06-30,7A,84894,Meluota Corp,2702 ASTORIA BLVD,ASTORIA,NY,11102,"Santander Bank, National Association",29950.0,...,CORPORATION,Existing or more than 2 years old,P I F,2023-11-30,NaN,0.0,Y,3.0,N,NaN
2,2026-06-30,7A,53803,THOMAS W CHASE,1700 GIDGET LN,COLFAX,CA,95713,"U.S. Bank, National Association",6548.0,...,INDIVIDUAL,Unanswered,P I F,2024-11-30,NaN,0.0,N,0.0,Y,NaN
3,2026-06-30,7A,123499,"513 SOLUTIONS GROUP, LLC",120 FIREBIRD RUN,CIBOLO,TX,78108,BayFirst National Bank,34997.0,...,CORPORATION,Existing or more than 2 years old,EXEMPT,NaN,NaN,0.0,N,3.0,N,Y
4,2026-06-30,7A,70400,"Tian Shan International, LLC",2503 BAGBY ST,HOUSTON,TX,77006,Golden Bank National Association,26223.0,...,CORPORATION,Unanswered,P I F,2024-01-31,NaN,0.0,N,4.0,Y,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
388333,2026-06-30,7A,29805,Gopinathg LLC,250 EAST WHITE HORSE PIKE,GALLOWAY,NJ,8205,"TD Bank, National Association",18409.0,...,CORPORATION,Existing or more than 2 years old,COMMIT,NaN,NaN,0.0,Y,2.0,Y,NaN
388334,2026-06-30,7A,455644,"Gallomar, LLC",297 W. ROUND GROVE RD,LEWISVILLE,TX,75067,Live Oak Banking Company,58665.0,...,CORPORATION,Existing or more than 2 years old,COMMIT,NaN,NaN,0.0,N,7.0,Y,NaN
388335,2026-06-30,7A,46391,A GOOD DEAL IN NEW JERSEY LLC,11 GLORIA LN,FAIRFIELD,NJ,7004,Manufacturers and Traders Trust Company,588.0,...,CORPORATION,Existing or more than 2 years old,COMMIT,NaN,NaN,0.0,Y,8.0,N,NaN
388336,2026-06-30,7A,455644,Read Enterprises LLC,115 Kilgore Drive,BRISTOL,VA,24201,Live Oak Banking Company,58665.0,...,CORPORATION,"Startup, Loan Funds will Open Business",COMMIT,NaN,NaN,0.0,N,3.0,Y,NaN


In [2]:
df.shape

(388338, 42)

In [3]:
df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 388338 entries, 0 to 388337
Data columns (total 42 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   AsOfDate                    388338 non-null  str    
 1   Program                     388338 non-null  str    
 2   LocationID                  388338 non-null  int64  
 3   BorrName                    388336 non-null  str    
 4   BorrStreet                  388338 non-null  str    
 5   BorrCity                    388338 non-null  str    
 6   BorrState                   388338 non-null  str    
 7   BorrZip                     388338 non-null  int64  
 8   BankName                    388338 non-null  str    
 9   BankFDICNumber              347016 non-null  float64
 10  BankNCUANumber              11147 non-null   float64
 11  BankStreet                  388338 non-null  str    
 12  BankCity                    388338 non-null  str    
 13  BankState                

## **Defining the target variable**

Before anything else, I need to decide what counts as a default. The file has five status codes, but only two of them tell you how a loan actually ended, paid in full, or charged off. Cancelled and committed loans never started repaying, and EXEMPT doesn't say what happened at all. None of those three can be labelled good or bad, so the model only uses the two that can.

In [4]:
# Counting how many loans fall into each LoanStatus category, most common first
df['LoanStatus'].value_counts()

LoanStatus
EXEMPT    242061
P I F      68201
CANCLD     50104
COMMIT     21079
CHGOFF      6893
Name: count, dtype: int64

In [5]:
# Keeping only loans with a completed, observable outcome: paid in full or charged off
model_df = df[df['LoanStatus'].isin(['P I F', 'CHGOFF'])]

# Confirming the filter worked as expected
model_df['LoanStatus'].value_counts()

LoanStatus
P I F     68201
CHGOFF     6893
Name: count, dtype: int64

In [6]:
# I'm filtering to only completed loans first, then locking in a copy so pandas
# doesn't complain when I add a new column to it
model_df = model_df.copy()

# I'm defining my target variable here: 1 means the loan was charged off (a default),
# 0 means it was paid in full. This 0/1 convention is standard in credit risk work,
# and I need to get the direction right since every metric later assumes it.
model_df['default'] = model_df['LoanStatus'].map({'P I F': 0, 'CHGOFF': 1})

# Quick sanity check: this should show exactly 68201 zeros and 6893 ones,
# matching the counts I found earlier with value_counts()
model_df['default'].value_counts()

default
0    68201
1     6893
Name: count, dtype: int64

# **Stage 2: Cleaning**

Most of the blanks in this file aren't really missing data...

## **Missing values**

Most of the blanks in this file aren't really missing data. A field is empty because it doesn't apply to that loan, a business that isn't a franchise has no franchise code. Those get left alone. Only the fields with real gaps get dropped, and those affect well unCader 1% of rows.

In [7]:
# Droping rows missing BusinessAge, InitialInterestRate, or CongressionalDistrict —
model_df = model_df.dropna(subset=['BusinessAge', 'InitialInterestRate', 'CongressionalDistrict'])

model_df.shape  # confirm how many rows this actually removed

(74981, 43)

In [8]:
df

,AsOfDate,Program,LocationID,BorrName,BorrStreet,BorrCity,BorrState,BorrZip,BankName,BankFDICNumber,...,BusinessType,BusinessAge,LoanStatus,PaidInFullDate,ChargeOffDate,GrossChargeOffAmount,RevolverStatus,JobsSupported,CollateralInd,SoldSecMrktInd
0,2026-06-30,7A,29805,"AMERIPRO CONSTRUCTION SERVICES, INC.",1403 SENTRY LANE,Norristown,PA,19403,"TD Bank, National Association",18409.0,...,PARTNERSHIP,Unanswered,P I F,2024-11-30,NaN,0.0,Y,5.0,N,NaN
1,2026-06-30,7A,84894,Meluota Corp,2702 ASTORIA BLVD,ASTORIA,NY,11102,"Santander Bank, National Association",29950.0,...,CORPORATION,Existing or more than 2 years old,P I F,2023-11-30,NaN,0.0,Y,3.0,N,NaN
2,2026-06-30,7A,53803,THOMAS W CHASE,1700 GIDGET LN,COLFAX,CA,95713,"U.S. Bank, National Association",6548.0,...,INDIVIDUAL,Unanswered,P I F,2024-11-30,NaN,0.0,N,0.0,Y,NaN
3,2026-06-30,7A,123499,"513 SOLUTIONS GROUP, LLC",120 FIREBIRD RUN,CIBOLO,TX,78108,BayFirst National Bank,34997.0,...,CORPORATION,Existing or more than 2 years old,EXEMPT,NaN,NaN,0.0,N,3.0,N,Y
4,2026-06-30,7A,70400,"Tian Shan International, LLC",2503 BAGBY ST,HOUSTON,TX,77006,Golden Bank National Association,26223.0,...,CORPORATION,Unanswered,P I F,2024-01-31,NaN,0.0,N,4.0,Y,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
388333,2026-06-30,7A,29805,Gopinathg LLC,250 EAST WHITE HORSE PIKE,GALLOWAY,NJ,8205,"TD Bank, National Association",18409.0,...,CORPORATION,Existing or more than 2 years old,COMMIT,NaN,NaN,0.0,Y,2.0,Y,NaN
388334,2026-06-30,7A,455644,"Gallomar, LLC",297 W. ROUND GROVE RD,LEWISVILLE,TX,75067,Live Oak Banking Company,58665.0,...,CORPORATION,Existing or more than 2 years old,COMMIT,NaN,NaN,0.0,N,7.0,Y,NaN
388335,2026-06-30,7A,46391,A GOOD DEAL IN NEW JERSEY LLC,11 GLORIA LN,FAIRFIELD,NJ,7004,Manufacturers and Traders Trust Company,588.0,...,CORPORATION,Existing or more than 2 years old,COMMIT,NaN,NaN,0.0,Y,8.0,N,NaN
388336,2026-06-30,7A,455644,Read Enterprises LLC,115 Kilgore Drive,BRISTOL,VA,24201,Live Oak Banking Company,58665.0,...,CORPORATION,"Startup, Loan Funds will Open Business",COMMIT,NaN,NaN,0.0,N,3.0,Y,NaN


In [9]:
model_df

,AsOfDate,Program,LocationID,BorrName,BorrStreet,BorrCity,BorrState,BorrZip,BankName,BankFDICNumber,...,BusinessAge,LoanStatus,PaidInFullDate,ChargeOffDate,GrossChargeOffAmount,RevolverStatus,JobsSupported,CollateralInd,SoldSecMrktInd,default
0,2026-06-30,7A,29805,"AMERIPRO CONSTRUCTION SERVICES, INC.",1403 SENTRY LANE,Norristown,PA,19403,"TD Bank, National Association",18409.0,...,Unanswered,P I F,2024-11-30,NaN,0.0,Y,5.0,N,NaN,0
1,2026-06-30,7A,84894,Meluota Corp,2702 ASTORIA BLVD,ASTORIA,NY,11102,"Santander Bank, National Association",29950.0,...,Existing or more than 2 years old,P I F,2023-11-30,NaN,0.0,Y,3.0,N,NaN,0
2,2026-06-30,7A,53803,THOMAS W CHASE,1700 GIDGET LN,COLFAX,CA,95713,"U.S. Bank, National Association",6548.0,...,Unanswered,P I F,2024-11-30,NaN,0.0,N,0.0,Y,NaN,0
4,2026-06-30,7A,70400,"Tian Shan International, LLC",2503 BAGBY ST,HOUSTON,TX,77006,Golden Bank National Association,26223.0,...,Unanswered,P I F,2024-01-31,NaN,0.0,N,4.0,Y,NaN,0
8,2026-06-30,7A,58036,"L. N. Zimmermann, Inc.",7790 Mainland Dr Ste 101,San Antonio,TX,78250,Fifth Third Bank,6672.0,...,Existing or more than 2 years old,P I F,2021-09-30,NaN,0.0,Y,0.0,Y,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373626,2026-06-30,7A,53803,OUTRIDER SALES CORPORATION,1650 OUTRIDER WAY,MONUMENT,CO,80132,"U.S. Bank, National Association",6548.0,...,New Business or 2 years or less,P I F,2026-05-31,NaN,0.0,N,4.0,Y,NaN,0
374130,2026-06-30,7A,49874,"MOSQUITONIX ANN ARBOR, BLOOMFIELD, BRIGHTON,NOVI",6955 RAVINES CIR,WEST BLOOMFIELD TOWNSHIP,MI,48322,"Newtek Bank, National Association",18734.0,...,"Startup, Loan Funds will Open Business",P I F,2026-05-31,NaN,0.0,N,4.0,Y,NaN,0
375000,2026-06-30,7A,53803,TA CONSTRUCTION CONSULTANTS LLC,9241 HUGHES DR,CORONA,CA,92883,"U.S. Bank, National Association",6548.0,...,Existing or more than 2 years old,P I F,2026-05-31,NaN,0.0,N,0.0,Y,NaN,0
375201,2026-06-30,7A,53803,DOUGLAS HERNANDEZ,156 EATON RD,CHICO,CA,95973,"U.S. Bank, National Association",6548.0,...,Existing or more than 2 years old,P I F,2026-05-31,NaN,0.0,N,8.0,Y,NaN,0


In [10]:
# ApprovalDate is currently just text, so I can't do date math on it (like finding how many days old a loan is). This converts it to a real date type.
model_df['ApprovalDate'] = pd.to_datetime(model_df['ApprovalDate'])

model_df['ApprovalDate'].dtype  # should now say datetime64, not object/str

dtype('<M8[us]')

In [11]:
# Pull just the approval year out of each date, so I can compare PIF vs CHGOFF by vintage
model_df['ApprovalYear'] = model_df['ApprovalDate'].dt.year

# Compare the approval-year distribution for paid-in-full loans vs charged-off loans
model_df.groupby('default')['ApprovalYear'].describe()

,count,mean,std,min,25%,50%,75%,max
default,,,,,,,,
0,68153.0,2021.211597,1.464269,2019.0,2020.0,2021.0,2022.0,2026.0
1,6828.0,2021.610428,1.379490,2019.0,2021.0,2022.0,2023.0,2025.0


In [12]:
# Comparing loan term length for paid-in-full vs charged-off loans
# checking if PIF loans are biased toward shorter terms just because they've had time to fully repay
model_df.groupby('default')['TermInMonths'].describe()


,count,mean,std,min,25%,50%,75%,max
default,,,,,,,,
0,68153.0,133.558625,78.759480,0.0,84.0,120.0,120.0,336.0
1,6828.0,87.157147,33.636004,0.0,69.0,92.0,108.0,300.0


In [13]:
# Grouping loan term stats by outcome, rounded to 1 decimal for readability
result = model_df.groupby('default')['TermInMonths'].describe().round(1)

# Relabeling 0/1 as readable text for display only, the actual 'default' column stays numeric for modelling
result.index = result.index.map({0: 'Paid in Full', 1: 'Charged Off'})

result

,count,mean,std,min,25%,50%,75%,max
default,,,,,,,,
Paid in Full,68153.0,133.6,78.8,0.0,84.0,120.0,120.0,336.0
Charged Off,6828.0,87.2,33.6,0.0,69.0,92.0,108.0,300.0


In [14]:
# Getting approval year for every loan in the full dataset (not just resolved ones)
df['ApprovalYear'] = df['ApprovalDate'].astype('datetime64[ns]').dt.year

# Comparing approval-year distribution across ALL LoanStatus categories, including still-active EXEMPT loans
df.groupby('LoanStatus')['ApprovalYear'].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
LoanStatus,,,,,,,,
CANCLD,50104.0,2023.1,2.0,2019.0,2021.0,2024.0,2025.0,2026.0
CHGOFF,6893.0,2021.6,1.4,2019.0,2020.0,2022.0,2023.0,2025.0
COMMIT,21079.0,2025.1,1.2,2019.0,2024.0,2026.0,2026.0,2026.0
EXEMPT,242061.0,2023.3,1.7,2019.0,2022.0,2024.0,2025.0,2026.0
P I F,68201.0,2021.2,1.5,2019.0,2020.0,2021.0,2022.0,2026.0


In [15]:
# Checking what's still missing in my cleaned working dataset
model_df.isnull().sum()

AsOfDate                          0
Program                           0
LocationID                        0
BorrName                          0
BorrStreet                        0
BorrCity                          0
BorrState                         0
BorrZip                           0
BankName                          0
BankFDICNumber                 6718
BankNCUANumber                72436
BankStreet                        0
BankCity                          0
BankState                         0
BankZip                           0
GrossApproval                     0
SBAGuaranteedApproval             0
ApprovalDate                      0
ApprovalFY                        0
FirstDisbursementDate             3
ProcessingMethod                  0
InitialInterestRate               0
FixedorVariableInterestInd        0
TermInMonths                      0
NaicsCode                         0
NaicsDescription                  0
FranchiseCode                 65576
FranchiseName               

In [16]:
# Droping the handful of rows still missing FirstDisbursementDate or BusinessType
model_df = model_df.dropna(subset=['FirstDisbursementDate', 'BusinessType'])

model_df.shape  # confirming final row count

(74976, 44)

## **Grouping NAICS into broad industry sectors**

Industry is likely to matter for default risk, but the NAICS codes in this file are six digits deep. That's far too granular to model, most individual codes would have only a handful of loans. Checking how many there actually are before deciding how to group them.

In [17]:
# How many distinct 6-digit NAICS codes exist in the data
model_df['NaicsCode'].nunique()

1037

In [18]:
# Convert NAICS to text and keep only the first 2 digits, this gives the broad industry sector
model_df['NaicsSector'] = model_df['NaicsCode'].astype(str).str[:2]

# How many broad sectors does that leave us with?
model_df['NaicsSector'].nunique()

24

In [19]:
# See the actual categories in BusinessType before deciding how to encode it
model_df['BusinessType'].unique()

<ArrowStringArray>
['PARTNERSHIP', 'CORPORATION', 'INDIVIDUAL']
Length: 3, dtype: str

## **Checking what's actually in BusinessAge**

Before deciding how to encode this field, it's worth seeing what values it actually contains.

In [20]:
# To see the actual categories in BusinessAge before deciding how to encode it
model_df['BusinessAge'].unique()

<ArrowStringArray>
[                            'Unanswered',
      'Existing or more than 2 years old',
 'Startup, Loan Funds will Open Business',
                    'Change of Ownership',
        'New Business or 2 years or less']
Length: 5, dtype: str

## **A missing category that standard checks don't catch**

'Unanswered' is missing data wearing a disguise, it passes every null check while telling you nothing about the business. Before dropping those rows, it's worth checking where they come from.

In [21]:
# How many rows have 'Unanswered' as their BusinessAge, rather than a real answer?
(model_df['BusinessAge'] == 'Unanswered').sum()

np.int64(2472)

In [22]:
# Does 'Unanswered' BusinessAge cluster around a specific processing method?
model_df[model_df['BusinessAge'] == 'Unanswered']['ProcessingMethod'].value_counts()

ProcessingMethod
SBA Express Program               1874
Preferred Lenders Program          479
7a General                          80
Community Advantage Initiative      26
Working Capital CAPLine              7
Export Express                       4
International Trade Loans            1
Seasonal CAPLine                     1
Name: count, dtype: int64

## **Feature engineering**

One new field here: a franchise flag, built from whether the franchise code is filled in. A blank means the business isn't a franchise, which is information worth keeping rather than discarding.

In [23]:
# FranchiseCode is null when a business isn't a franchise, so this turns that into a usable 0/1 feature
model_df['IsFranchise'] = model_df['FranchiseCode'].notna().astype(int)

model_df['IsFranchise'].value_counts()  # checking the split between franchise and non-franchise loans

IsFranchise
0    65571
1     9405
Name: count, dtype: int64

In [24]:
model_df.shape


(74976, 46)

**Saving the clean dataset** to use in further analysis ahead

In [25]:
model_df.to_csv('Model df cleaned.csv', index=False)